In [53]:
!pip install langchain chromadb faiss-cpu tiktoken pypdf langchain-community langchain_huggingface wikipedia

## Wikipedia Retriever

In [54]:
from langchain_community.retrievers import WikipediaRetriever

In [55]:
import os
import wikipedia

os.environ["USER_AGENT"] = "GenAI-LangChain-learning/1.0 (subhamomm836@gmail.com)"
wikipedia.set_user_agent("GenAI-LangChain-learning/1.0 (subhamomm836@gmail.com)")

In [56]:
retriever=WikipediaRetriever(top_k_results=2,lang="en")

In [57]:
query="The geopolitical history of India and Pakistan from the perspective of a chinese"

docs=retriever.invoke(query)

In [58]:
for i, doc in enumerate(docs):
    print(f"\n Document {i+1}:")
    print(f"Content: \n {doc.page_content}")
    print("\n")


 Document 1:
Content: 
 The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,00

## Vector Store Retriever

In [59]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from langchain_core.documents import Document

In [60]:
# create documents
documents=[
    Document(page_content="Lnagchain helps developer build LLM application easily"),
    Document(page_content="Chroma is a vector database optimized for LLM based search"),
    Document(page_content="embeddings convert text into high dientional vectors"),
    Document(page_content="OpenAI provides powerful embedding models")
]

In [61]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [62]:
# initialize embedding model
embedding = HuggingFaceEndpointEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2',
    )

In [63]:
# create vector store
vector_store=Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    collection_name='sample'
)

In [64]:
# convert vector store into a retriever
retriever=vector_store.as_retriever(search_kwargs={"k":2})

In [65]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [66]:
for i, doc in enumerate(results):
  print(f"\n Result {i+1}:")
  print(f"Content: \n {doc.page_content}")
  print("\n")


 Result 1:
Content: 
 Chroma is a vector database optimized for LLM based search



 Result 2:
Content: 
 Chroma is a vector database optimized for LLM based search




## MMR

In [67]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [68]:
from langchain_community.vectorstores import FAISS

In [69]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [70]:
# initialize embedding model
embedding = HuggingFaceEndpointEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2',
    )

In [71]:
# create vector store
vector_store=FAISS.from_documents(
    documents=docs,
    embedding=embedding,
)

In [72]:
# convert vector store into a retriever
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3,"lambda_mult":0} # lambda_mult=1: simple similarity search, lambda_mult=0: diverse results
    )

In [73]:
query="What is langchain?"
results=retriever.invoke(query)

In [74]:
for i,doc in enumerate(results):
  print(f"\n Result {i+1}:")
  print(f"Content: \n {doc.page_content}")
  print("\n")


 Result 1:
Content: 
 LangChain supports Chroma, FAISS, Pinecone, and more.



 Result 2:
Content: 
 MMR helps you get diverse results when doing similarity search.



 Result 3:
Content: 
 Embeddings are vector representations of text.




## Multi Query Retriever

In [75]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [76]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [77]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [78]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [79]:
# initialize embedding model
embedding = HuggingFaceEndpointEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2',
    )

In [80]:
# create vector store
vector_store=FAISS.from_documents(
    documents=all_docs,
    embedding=embedding,
)

In [81]:
# convert vector store into a retriever
similarity_retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
    )

In [82]:
multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k":5}),
    llm=model
)

In [83]:
# Query
query = "How to improve energy levels and maintain balance?"

In [84]:
# Retrieve results
similarity_results=similarity_retriever.invoke(query)
multiquery_results=multiquery_retriever.invoke(query)

In [85]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental 

## Contextual Compression Retriever

In [97]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [98]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [99]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [100]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [101]:
# initialize embedding model
embedding = HuggingFaceEndpointEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2',
    )

In [102]:
# create vector store
vector_store=FAISS.from_documents(
    documents=all_docs,
    embedding=embedding,
)

In [103]:
# base retriever
base_retriever=vector_store.as_retriever(
    search_kwargs={"k":2}
    )

In [104]:
# setip compressor using an LLM
compressor=LLMChainExtractor.from_llm(
    llm=model
)

In [105]:
compression_retriever=ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [106]:
query="What is photosynthesis?"
results=compression_retriever.invoke(query)

In [107]:
for i,doc in enumerate(results):
  print(f"\n Result {i+1}:")
  print(f"Content: \n {doc.page_content}")
  print("\n")


 Result 1:
Content: 
 Photosynthesis enables plants to produce energy by converting sunlight.


